## Python Challenges: Introduction to machine learning
---
This notebook provides scenario-based exercises to reinforce your learning on `sklearn` estimators, transformers, and pipelines. 

As covered in the workshop, `sklearn` stores and outputs nearly everything as a **numpy array**, but your dataset features are typically represented by a Pandas DataFrame. You will practice standardizing numerical features, encoding categorical features, and fitting models using both step-by-step methods and the `Pipeline` class. 

In [ ]:
# Import necessary libraries to set up the notebook

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

### Generating Synthetic Data for the Exercises
Run the cell below to generate the synthetic datasets we will use for the two scenarios.

In [ ]:
# Dataset 1: Real Estate (For Exercise 1)
np.random.seed(42)
n_samples = 500
size = np.random.normal(1500, 500, n_samples)
beds = np.random.randint(1, 6, n_samples)
age = np.random.randint(1, 50, n_samples)
neighborhood = np.random.choice(['Urban', 'Suburban', 'Rural'], n_samples)

# Formulate price: Base + Size factor + Beds factor - Age factor + Neighborhood factor + Noise
price = (size * 200) + (beds * 10000) - (age * 1500) + np.where(neighborhood=='Urban', 50000, np.where(neighborhood=='Suburban', 20000, 0)) + np.random.normal(0, 15000, n_samples)

df_real_estate = pd.DataFrame({
    'Size_sqft': size, 
    'Bedrooms': beds, 
    'Age_years': age, 
    'Neighborhood_Type': neighborhood, 
    'Price': price
})

# Dataset 2: HR Analytics (For Exercise 2)
exp = np.random.uniform(1, 25, n_samples)
perf = np.random.uniform(1, 5, n_samples)
dept = np.random.choice(['Sales', 'Engineering', 'Marketing', 'HR'], n_samples)
salary = 40000 + (exp * 3000) + (perf * 5000) + np.where(dept=='Engineering', 20000, np.where(dept=='Sales', 5000, 0)) + np.random.normal(0, 5000, n_samples)

df_hr = pd.DataFrame({
    'Experience': exp, 
    'Performance': perf, 
    'Department': dept, 
    'Salary': salary
})

---
## Exercise 1: Real Estate Price Prediction

**Scenario**: You are a Data Scientist at a modern real estate agency. Your goal is to build a regression model that accurately predicts house prices based on historical sales data. Linear regression fits a line $y = mx + c$ to multidimensional data. You must ensure the data is properly preprocessed to avoid giving incorrect weight to features with larger scales.

**Tasks**:
1. Separate features (`X`) and the target variable (`y`, which is `Price`).
2. Perform a train-test split setting 20% of the data aside for testing.
3. Apply `StandardScaler` to the numerical features to center the data around 0 and scale it to have a standard deviation of 1. 
4. Apply `OneHotEncoder` (with `sparse_output=False` and `drop='first'`) to the categorical features to avoid perfect multicollinearity.
5. Fit a `LinearRegression` model and evaluate it using Mean Absolute Error (MAE) and the $R^2$ score on the test set.

In [ ]:
# Step 1 & 2: Separate X and y, and split the data

In [ ]:
# Define column types

In [ ]:
# Step 3: Scale numerical features

In [ ]:
# Step 4: Encode categorical features

In [ ]:
# Combine processed features

In [ ]:
# Step 5: Fit and Evaluate

In [ ]:
# SOLUTION

# Step 1 & 2: Separate X and y, and split the data
X_re = df_real_estate.drop('Price', axis=1)
y_re = df_real_estate['Price']

X_train_re, X_test_re, y_train_re, y_test_re = train_test_split(X_re, y_re, test_size=0.2, random_state=42)

# Define column types
num_cols_re = ['Size_sqft', 'Bedrooms', 'Age_years']
cat_cols_re = ['Neighborhood_Type']

# Step 3: Scale numerical features
scaler = StandardScaler()
scaled_train = scaler.fit_transform(X_train_re[num_cols_re])
scaled_test = scaler.transform(X_test_re[num_cols_re])

scaled_train_df = pd.DataFrame(scaled_train, columns=num_cols_re, index=X_train_re.index)
scaled_test_df = pd.DataFrame(scaled_test, columns=num_cols_re, index=X_test_re.index)

# Step 4: Encode categorical features
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_train = encoder.fit_transform(X_train_re[cat_cols_re])
encoded_test = encoder.transform(X_test_re[cat_cols_re])

new_cat_cols = encoder.get_feature_names_out(cat_cols_re)
encoded_train_df = pd.DataFrame(encoded_train, columns=new_cat_cols, index=X_train_re.index)
encoded_test_df = pd.DataFrame(encoded_test, columns=new_cat_cols, index=X_test_re.index)

# Combine processed features
X_train_processed_re = pd.concat([encoded_train_df, scaled_train_df], axis=1)
X_test_processed_re = pd.concat([encoded_test_df, scaled_test_df], axis=1)

# Step 5: Fit and Evaluate
model_re = LinearRegression()
model_re.fit(X_train_processed_re, y_train_re)

y_pred_re = model_re.predict(X_test_processed_re)
print(f"Test MAE: {mean_absolute_error(y_test_re, y_pred_re):.2f}")
print(f"Test R^2: {r2_score(y_test_re, y_pred_re):.4f}")

### Optional Stretch: Using `sklearn` Pipelines
The pipeline object takes a list of `("name", "estimator")` tuples as input, where `name` is a string used to label the step. Re-implement the above preprocessing and modeling workflow more conveniently using a `Pipeline` and `ColumnTransformer` to enforce the execution and order of steps.

In [ ]:
# Write your code here

# Fit pipeline


# Predict and evaluate

In [ ]:

# Stretch Solution: Pipeline
preprocessor_re = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols_re),
    ('cat', OneHotEncoder(sparse_output=False, drop='first'), cat_cols_re)
])

pipeline_re = Pipeline(steps=[
    ('preprocessor', preprocessor_re),
    ('model', LinearRegression())
])

# Fit pipeline
pipeline_re.fit(X_train_re, y_train_re)

# Predict and evaluate
y_pred_pipe = pipeline_re.predict(X_test_re)
print(f"Pipeline Test MAE: {mean_absolute_error(y_test_re, y_pred_pipe):.2f}")
print(f"Pipeline Test R^2: {r2_score(y_test_re, y_pred_pipe):.4f}")

---
## Exercise 2: Salary prediction

**Scenario**: As a data scientist in Department of Work and Pension, you want to investigate whether pay equity across departments has improved. You are building an associative model to understand the relationship between data features (inputs) and the target (output) to make predictions for new instances. You will use the `df_hr` dataset.

**Tasks**:
1. Train-test split your data (Target is `Salary`).
2. Build a pipeline that scales numerical features (`Experience`, `Performance`) and encodes categorical features (`Department`).
3. Fit a linear regression model using this pipeline and evaluate its performance.

In [ ]:
# Step 1: Split the data
# write your code here

In [ ]:
# Step 2 & 3: Build Pipeline, Fit, and Evaluate
# Write your code here


In [ ]:
# SOLUTIONS - for reference only, you can compare your code with this after you have attempted the exercises on your own

# Step 1: Split the data
X_hr = df_hr.drop('Salary', axis=1)
y_hr = df_hr['Salary']

X_train_hr, X_test_hr, y_train_hr, y_test_hr = train_test_split(X_hr, y_hr, test_size=0.2, random_state=100)

num_cols_hr = ['Experience', 'Performance']
cat_cols_hr = ['Department']

# Step 2 & 3: Build Pipeline, Fit, and Evaluate
preprocessor_hr = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols_hr),
    ('cat', OneHotEncoder(sparse_output=False, drop='first'), cat_cols_hr)
])

pipeline_hr = Pipeline(steps=[
    ('preprocessor', preprocessor_hr),
    ('model', LinearRegression())
])

pipeline_hr.fit(X_train_hr, y_train_hr)
y_pred_hr = pipeline_hr.predict(X_test_hr)

print(f"HR Salary Prediction - Test MAE: {mean_absolute_error(y_test_hr, y_pred_hr):.2f}")
print(f"HR Salary Prediction - Test R^2: {r2_score(y_test_hr, y_pred_hr):.4f}")

### Optional Stretch: Feature Selection with RFE
Recursive Feature Elimination (RFE) drops the least important feature at a time, recalculates, and repeats to narrow down the dataset. Modify your HR pipeline to use `RFE` to select only the top 3 most important features before passing data to the final model.

In [ ]:
# Write your pipeline code here, fit, predict, and evaluate



# Produce a report to view which features were selected


In [ ]:
# SOLUTION 

# Stretch Solution: Pipeline with RFE
pipeline_hr_rfe = Pipeline(steps=[
    ('preprocessor', preprocessor_hr),
    ('feature_selection', RFE(estimator=LinearRegression(), n_features_to_select=3, step=1)),
    ('model', LinearRegression())
])

pipeline_hr_rfe.fit(X_train_hr, y_train_hr)
y_pred_hr_rfe = pipeline_hr_rfe.predict(X_test_hr)

print(f"HR Salary (Top 3 Features) - Test MAE: {mean_absolute_error(y_test_hr, y_pred_hr_rfe):.2f}")
print(f"HR Salary (Top 3 Features) - Test R^2: {r2_score(y_test_hr, y_pred_hr_rfe):.4f}")

# View which features were selected
rfe_step = pipeline_hr_rfe.named_steps['feature_selection']
selected_indices = rfe_step.get_support(indices=True)
feature_names = preprocessor_hr.get_feature_names_out()
print(f"\nTop 3 selected features: {feature_names[selected_indices]}")